# Методы кластеризации scikit-learn
## Применение к данным стоматологической клиники

**Задание 4**  
**Платформа:** Jupyter Notebook  

---

### Применение к стоматологии

Данные моделируют записи приёмов стоматологической клиники:  
- **Стоимость приёма** и **количество зубов** → кластеры типов лечения  
- **Частота визитов** и **средняя стоимость** → профили пациентов  
- **Диагнозы и материалы** → группы схожих случаев

### Рассматриваемые методы:
1. **K-Means** — K-средних
2. **Affinity Propagation** — метод распространения близости
3. **Mean Shift** — средний сдвиг
4. **Spectral Clustering** — спектральная кластеризация
5. **Agglomerative (Hierarchical)** — иерархическая кластеризация
6. **DBSCAN** — кластеризация по плотности
7. **HDBSCAN** — иерархический DBSCAN
8. **OPTICS** — упорядочение точек для идентификации кластерной структуры
9. **BIRCH** — сбалансированные итеративные деревья
10. **Оценка производительности** — метрики качества кластеризации

## 1. Генерация данных стоматологической клиники

In [ ]:
np.random.seed(42)
N = 400

# ── Три типа пациентов / лечения ─────────────────────────────────────────────
# Группа 1: Профилактика (мало зубов, дешево)
g1 = np.random.multivariate_normal([1.2, 1500], [[0.15,200],[200,300000]], 140)
# Группа 2: Лечение кариеса / пульпит (средне)
g2 = np.random.multivariate_normal([2.5, 5500], [[0.2,100],[100,600000]], 160)
# Группа 3: Сложное лечение / импланты (много зубов, дорого)
g3 = np.random.multivariate_normal([3.8, 11000], [[0.2,50],[50,800000]], 100)

raw = np.vstack([g1, g2, g3])
true_labels = np.array([0]*140 + [1]*160 + [2]*100)

df = pd.DataFrame(raw, columns=['teeth_count', 'cost_rub'])
df['teeth_count'] = df['teeth_count'].clip(1, 8).round(1)
df['cost_rub']    = df['cost_rub'].clip(500, 18000).round(0)
df['true_group']  = true_labels

# Нормализация для алгоритмов
scaler = StandardScaler()
X = scaler.fit_transform(df[['teeth_count', 'cost_rub']])

print(f'Датасет: {len(df)} записей приёмов')
print(df.describe().round(1))

# Визуализация исходных данных
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(df['teeth_count'], df['cost_rub'],
               c=[PALETTE[i] for i in true_labels], alpha=0.6, s=40, edgecolors='white', lw=0.3)
axes[0].set_xlabel('Количество зубов в приёме', fontsize=11)
axes[0].set_ylabel('Стоимость приёма (₽)', fontsize=11)
axes[0].set_title('Исходные данные (истинные группы)', fontsize=12, fontweight='bold')
for i, label in enumerate(['Профилактика', 'Лечение кариеса', 'Сложное лечение']):
    axes[0].scatter([], [], c=PALETTE[i], label=label, s=60)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X[:,0], X[:,1],
               c=[PALETTE[i] for i in true_labels], alpha=0.6, s=40, edgecolors='white', lw=0.3)
axes[1].set_xlabel('Зубы (нормализовано)', fontsize=11)
axes[1].set_ylabel('Стоимость (нормализовано)', fontsize=11)
axes[1].set_title('Нормализованные данные (для алгоритмов)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle(' Данные стоматологической клиники', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## Метод 1: K-Means (K-средних)

### Математика
K-Means минимизирует **инерцию** — сумму квадратов расстояний от каждой точки до центроида её кластера:

$$J = \sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2$$

где $\mu_i$ — центроид кластера $C_i$.

### Алгоритм (lloyd's algorithm):
1. Случайно выбираем $k$ начальных центроидов (или метод **k-means++**)
2. **E-шаг:** каждую точку относим к ближайшему центроиду
3. **M-шаг:** пересчитываем центроиды как среднее точек в кластере
4. Повторяем шаги 2–3 до сходимости

### Параметры:
 * n_clusters — количество кластеров $k$ (задаётся вручную!)
 * init='k-means++ — умная инициализация центроидов
 * n_init=10` — количество перезапусков с разными начальными точками
 * max_iter=300` — максимум итераций

### Метод локтя (Elbow Method)
Помогает выбрать оптимальное $k$: ищем точку перегиба на графике инерции.

In [ ]:
# ── Метод локтя для выбора k ─────────────────────────────────────────────────
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# График инерции
axes[0].plot(list(K_range), inertias, 'o-', color=PALETTE[0], lw=2, ms=8)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Оптимальное k=3')
axes[0].set_xlabel('Количество кластеров k')
axes[0].set_ylabel('Инерция (Inertia)')
axes[0].set_title('Метод локтя (Elbow Method)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# График силуэта
axes[1].plot(list(K_range), silhouettes, 's-', color=PALETTE[1], lw=2, ms=8)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Количество кластеров k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Силуэтный коэффициент', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# K-Means с k=3
km3 = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_km = km3.fit_predict(X)
centers = km3.cluster_centers_

for i in range(3):
    mask = labels_km == i
    axes[2].scatter(X[mask,0], X[mask,1], c=PALETTE[i], alpha=0.6, s=40,
                   edgecolors='white', lw=0.3, label=f'Кластер {i+1}')
axes[2].scatter(centers[:,0], centers[:,1], c='black', marker='X', s=200,
               zorder=5, label='Центроиды')
axes[2].set_title('K-Means (k=3): результат', fontweight='bold')
axes[2].set_xlabel('Зубы (норм.)'); axes[2].set_ylabel('Стоимость (норм.)')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.suptitle(' K-Means на данных стоматологической клиники', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nK-Means (k=3) результаты:')
print(f' Инерция:         {km3.inertia_:.2f}')
print(f' Silhouette:      {silhouette_score(X, labels_km):.4f}')
print(f' Итераций:        {km3.n_iter_}')
print(f'Размеры кластеров: {dict(zip(*np.unique(labels_km, return_counts=True)))}')
print()
print(' Вывод: K-Means чётко выделил 3 группы:')
print('   Кластер 1 — профилактика (1-2 зуба, до 2000₽)')
print('   Кластер 2 — плановое лечение (2-3 зуба, 4000-7000₽)')
print('   Кластер 3 — сложное лечение (3-5 зубов, от 8000₽)')

---
## 📡 Метод 2: Affinity Propagation (Метод распространения близости)

### Математика
AP не требует задания числа кластеров. Каждая точка отправляет **сообщения** двух типов:

- **Responsibility** $r(i,k)$ — насколько точка $k$ подходит для роли «образцового представителя» (exemplar) для точки $i$
- **Availability** $a(i,k)$ — насколько уместно точке $i$ выбрать $k$ своим представителем

$$r(i,k) \leftarrow s(i,k) - \max_{k' \neq k}[a(i,k') + s(i,k')]$$
$$a(i,k) \leftarrow \min\left(0,\ r(k,k) + \sum_{i' \notin \{i,k\}} \max(0, r(i',k))\right)$$

где $s(i,k)$ — сходство (similarity) между точками $i$ и $k$.

### Параметры:
- `preference` — склонность точки быть представителем (чем выше → больше кластеров)
- `damping` — коэффициент затухания сообщений (0.5–1.0, предотвращает осцилляции)
- `max_iter` — максимум итераций обмена сообщениями

In [ ]:
# Affinity Propagation — работает лучше на небольших выборках
X_small = X[:120]  # берём подвыборку для скорости

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

preferences = [-5, -1, 1]
titles = ['preference=-5\n(мало кластеров)', 'preference=-1\n(средне)', 'preference=1\n(много кластеров)']

for ax, pref, title in zip(axes, preferences, titles):
    ap = AffinityPropagation(preference=pref, damping=0.7, random_state=42, max_iter=300)
    labels_ap = ap.fit_predict(X_small)
    n_clusters = len(ap.cluster_centers_indices_)
    
    colors = plt.cm.tab10(np.linspace(0, 0.9, n_clusters))
    for i in range(n_clusters):
        mask = labels_ap == i
        if mask.sum() > 0:
            ax.scatter(X_small[mask,0], X_small[mask,1],
                      color=colors[i], alpha=0.7, s=40, edgecolors='white', lw=0.3)
    # Exemplars (представители)
    exemplars = X_small[ap.cluster_centers_indices_]
    ax.scatter(exemplars[:,0], exemplars[:,1], c='black', marker='*', s=250,
              zorder=5, label=f'Exemplars ({n_clusters})')
    ax.set_title(f'AP: {title}\n→ {n_clusters} кластеров', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('📡 Affinity Propagation: влияние параметра preference', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Лучшая конфигурация
ap_best = AffinityPropagation(preference=-1, damping=0.7, random_state=42)
ap_best.fit(X_small)
n_cl = len(ap_best.cluster_centers_indices_)
print(f'\nAffinity Propagation (preference=-1):')
print(f'  Найдено кластеров: {n_cl}')
print(f'  Число итераций:   {ap_best.n_iter_}')
if n_cl > 1:
    print(f'  Silhouette:       {silhouette_score(X_small, ap_best.labels_):.4f}')
print()
print(' Особенность AP: сам определяет число кластеров.')
print('  В стоматологии: автоматически находит типичные "шаблонные" случаи')
print('  без предварительного знания о числе групп пациентов.')


## Метод 3: Mean Shift (Средний сдвиг)

### Математика
Mean Shift — непараметрический метод, основанный на оценке плотности. Каждая точка итерационно сдвигается в направлении **максимума плотности** своей окрестности:

$$m(x) = \frac{\sum_{x_i \in N(x)} K\left(\frac{x_i - x}{h}\right) x_i}{\sum_{x_i \in N(x)} K\left(\frac{x_i - x}{h}\right)} - x$$

где $K$ — ядро (обычно гауссовское), $h$ — ширина окна (bandwidth).

### Алгоритм:
1. Для каждой точки вычисляем среднее по точкам в шаровой окрестности радиуса $h$
2. Сдвигаем точку к этому среднему
3. Повторяем до сходимости → все точки "стягиваются" к локальным максимумам
4. Сгруппированные вместе точки → один кластер

### Параметры:
* `bandwidth` — ключевой параметр! Определяет радиус окрестности
* `estimate_bandwidth()` — автоматическая оценка bandwidth по данным

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

bandwidths = [0.3, 0.8, 1.5]
bw_auto = estimate_bandwidth(X, quantile=0.3, n_samples=200, random_state=42)

for ax, bw in zip(axes[:2], bandwidths[:2]):
    ms = MeanShift(bandwidth=bw, bin_seeding=True)
    labels_ms = ms.fit_predict(X)
    n_cl = len(np.unique(labels_ms))
    colors = plt.cm.tab10(np.linspace(0, 0.9, n_cl))
    for i in range(n_cl):
        mask = labels_ms == i
        ax.scatter(X[mask,0], X[mask,1], color=colors[i], alpha=0.6, s=35,
                  edgecolors='white', lw=0.3)
    ax.scatter(ms.cluster_centers_[:,0], ms.cluster_centers_[:,1],
              c='black', marker='X', s=180, zorder=5)
    ax.set_title(f'bandwidth={bw} → {n_cl} кластеров', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.grid(True, alpha=0.3)

# Автоматический bandwidth
ms_auto = MeanShift(bandwidth=bw_auto, bin_seeding=True)
labels_ms_auto = ms_auto.fit_predict(X)
n_cl_auto = len(np.unique(labels_ms_auto))
colors = plt.cm.tab10(np.linspace(0, 0.9, n_cl_auto))
for i in range(n_cl_auto):
    mask = labels_ms_auto == i
    axes[2].scatter(X[mask,0], X[mask,1], color=colors[i], alpha=0.6, s=35,
                   edgecolors='white', lw=0.3, label=f'K{i+1}')
axes[2].scatter(ms_auto.cluster_centers_[:,0], ms_auto.cluster_centers_[:,1],
               c='black', marker='X', s=200, zorder=5, label='Центры')
axes[2].set_title(f'auto bw={bw_auto:.2f} → {n_cl_auto} кластеров', fontweight='bold')
axes[2].set_xlabel('Зубы (норм.)'); axes[2].set_ylabel('Стоимость (норм.)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle(' Mean Shift: влияние bandwidth', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

if n_cl_auto > 1:
    sil = silhouette_score(X, labels_ms_auto)
    print(f'\nMean Shift (auto bandwidth={bw_auto:.3f}):')
    print(f'  Найдено кластеров: {n_cl_auto}')
    print(f'  Silhouette:        {sil:.4f}')
print()
print(' Mean Shift не требует k, но сильно зависит от bandwidth.')
print(' Хорош для нахождения «естественных» пиков плотности приёмов.')


## Метод 4: Спектральная кластеризация (Spectral Clustering)

Спектральная кластеризация использует **лапласиан графа** данных:

1. Строим **матрицу сходства** $W$ (affinity matrix) — обычно через гауссовское ядро:
$$W_{ij} = \exp\left(-\frac{\|x_i - x_j\|^2}{2\sigma^2}\right)$$

2. Вычисляем **нормализованный лапласиан**: $L = D^{-1/2}(D - W)D^{-1/2}$  
   где $D$ — диагональная матрица степеней

3. Находим **первые k собственных векторов** $L$

4. Применяем **K-Means** в пространстве собственных векторов

Идеален для данных сложной формы (кольца, полумесяцы), где евклидово расстояние не работает.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# ── Часть 1: на наших стоматологических данных ───────────────────────────────
sc = SpectralClustering(n_clusters=3, affinity='rbf', gamma=1.0,
                        assign_labels='kmeans', random_state=42, n_init=10)
labels_sc = sc.fit_predict(X)

for i in range(3):
    mask = labels_sc == i
    axes[0,0].scatter(X[mask,0], X[mask,1], c=PALETTE[i], alpha=0.6, s=40,
                     edgecolors='white', lw=0.3, label=f'Кластер {i+1}')
axes[0,0].set_title('Стоматология: Spectral (k=3)', fontweight='bold')
axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

# Матрица сходства
from sklearn.metrics.pairwise import rbf_kernel
W_small = rbf_kernel(X[:80], gamma=1.0)
im = axes[0,1].imshow(W_small, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=axes[0,1])
axes[0,1].set_title('Матрица сходства (affinity matrix)', fontweight='bold')
axes[0,1].set_xlabel('Индекс точки'); axes[0,1].set_ylabel('Индекс точки')

# Влияние gamma
gammas = [0.1, 1.0, 5.0]
for ax, g in zip(axes[0,2:] + axes[1,:2] if False else [axes[0,2]], [gammas[0]]):
    sc_g = SpectralClustering(n_clusters=3, affinity='rbf', gamma=g, random_state=42)
    lbl = sc_g.fit_predict(X)
    for i in range(3):
        mask = lbl == i
        ax.scatter(X[mask,0], X[mask,1], c=PALETTE[i], alpha=0.6, s=35,
                  edgecolors='white', lw=0.3)
    ax.set_title(f'gamma={g}', fontweight='bold'); ax.grid(True, alpha=0.3)

# ── Часть 2: демонстрация на нелинейных данных ───────────────────────────────
datasets = [
    make_moons(n_samples=200, noise=0.05, random_state=42),
    make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=42),
]
ds_names = ['Полумесяцы (make_moons)', 'Кольца (make_circles)']

for ax, (X_ds, y_ds), name in zip([axes[1,0], axes[1,1]], datasets, ds_names):
    X_ds_sc = StandardScaler().fit_transform(X_ds)
    sc_nl = SpectralClustering(n_clusters=2, affinity='rbf', gamma=5.0, random_state=42)
    lbl_nl = sc_nl.fit_predict(X_ds_sc)
    for i in range(2):
        mask = lbl_nl == i
        ax.scatter(X_ds_sc[mask,0], X_ds_sc[mask,1], c=PALETTE[i],
                  alpha=0.7, s=40, edgecolors='white', lw=0.3)
    ax.set_title(f'Spectral: {name}', fontweight='bold')
    ax.grid(True, alpha=0.3)

# K-Means на тех же нелинейных данных (для сравнения)
X_moon, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
X_moon_s = StandardScaler().fit_transform(X_moon)
km_nl = KMeans(n_clusters=2, random_state=42)
lbl_km_nl = km_nl.fit_predict(X_moon_s)
for i in range(2):
    mask = lbl_km_nl == i
    axes[1,2].scatter(X_moon_s[mask,0], X_moon_s[mask,1], c=PALETTE[i],
                     alpha=0.7, s=40, edgecolors='white', lw=0.3)
axes[1,2].set_title('K-Means на полумесяцах\n(для сравнения — работает хуже!)', fontweight='bold')
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('🔮 Спектральная кластеризация', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

sil_sc = silhouette_score(X, labels_sc)
print(f'\nSpectral Clustering (k=3):')
print(f'  Silhouette: {sil_sc:.4f}')
print()
print(' Спектральная кластеризация работает там, где K-Means провалится:')
print(' нелинейные границы, кольцеобразные структуры.')
print(' В стоматологии: выявление сложно разделимых групп пациентов.')

---
## Метод 5: Иерархическая кластеризация (Agglomerative)

Агломеративная (снизу вверх) иерархическая кластеризация:

1. Каждая точка — отдельный кластер
2. На каждом шаге **объединяем** два наиболее близких кластера
3. Повторяем до одного кластера → строим **дендрограмму**

### Критерии связи (linkage):
- **Ward** (минимизация дисперсии внутри кластера) — лучший для компактных кластеров
- **Complete** (максимальное расстояние между точками) — нечувствителен к выбросам  
- **Average** (среднее расстояние)
- **Single** (минимальное расстояние) — цепной эффект

### Дендрограмма
Визуализирует иерархию слияний. Горизонтальный разрез → разбиение на кластеры.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig)

# ── Дендрограмма (на подвыборке для читаемости) ───────────────────────────────
X_dendro = X[:60]
Z = linkage(X_dendro, method='ward')
ax_dendro = fig.add_subplot(gs[0, :])
dendrogram(Z, ax=ax_dendro, truncate_mode='level', p=5,
           leaf_rotation=45, leaf_font_size=8,
           color_threshold=3.5,
           above_threshold_color='gray')
ax_dendro.axhline(y=3.5, color='red', linestyle='--', alpha=0.8, label='Порог разреза (3 кластера)')
ax_dendro.set_title('Дендрограмма (Ward linkage, первые 60 точек)', fontweight='bold', fontsize=12)
ax_dendro.set_xlabel('Индексы точек (или кластеры)')
ax_dendro.set_ylabel('Расстояние')
ax_dendro.legend(); ax_dendro.grid(True, alpha=0.2)

# ── Сравнение linkage методов ────────────────────────────────────────────────
linkages = ['ward', 'complete', 'average']
linkage_names = ['Ward', 'Complete', 'Average']

for ax, link, name in zip([fig.add_subplot(gs[1, i]) for i in range(3)], linkages, linkage_names):
    agg = AgglomerativeClustering(n_clusters=3, linkage=link)
    labels_agg = agg.fit_predict(X)
    sil = silhouette_score(X, labels_agg)
    for i in range(3):
        mask = labels_agg == i
        ax.scatter(X[mask,0], X[mask,1], c=PALETTE[i], alpha=0.6, s=35,
                  edgecolors='white', lw=0.3, label=f'K{i+1} (n={mask.sum()})')
    ax.set_title(f'{name} linkage\nSilhouette={sil:.3f}', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle(' Иерархическая кластеризация (Agglomerative)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Лучший вариант — Ward
agg_ward = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg_ward.fit_predict(X)
print(f'\nAgglomerative (Ward, k=3):')
print(f'  Silhouette:  {silhouette_score(X, labels_agg):.4f}')
print(f'  Calinski-Harabasz: {calinski_harabasz_score(X, labels_agg):.2f}')
print()
print('   Ward-метод даёт компактные кластеры. Дендрограмма позволяет')
print('   визуально выбирать число кластеров без запуска нового алгоритма.')

---
## Метод 6: DBSCAN (Density-Based Spatial Clustering)

### Математика
DBSCAN делит точки на три типа на основе **плотности** окрестности:

- **Core point** (ядровая точка): в радиусе $\varepsilon$ от неё не менее $minPts$ точек
- **Border point** (граничная): в окрестности ядровой, но сама не ядровая
- **Noise** (шум/выброс): не ядровая и не граничная → метка $-1$

**Плотная достижимость:** $q$ достижима из $p$, если существует цепочка ядровых точек $p_1, p_2, ..., p_n$, где каждая в $\varepsilon$-окрестности следующей.

### Параметры:
- `eps` ($\varepsilon$) — радиус окрестности
- `min_samples` ($minPts$) — минимум точек для ядровой

### Преимущества:
- Находит кластеры **произвольной формы**
- Автоматически определяет **выбросы**
- Не нужно знать число кластеров

In [ ]:
# k-distance plot для подбора eps
from sklearn.neighbors import NearestNeighbors

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# k-distance graph
nbrs = NearestNeighbors(n_neighbors=5).fit(X)
distances, _ = nbrs.kneighbors(X)
k_dist = np.sort(distances[:, 4])[::-1]
axes[0,0].plot(k_dist, color=PALETTE[0], lw=2)
axes[0,0].axhline(y=0.3, color='red', linestyle='--', label='eps ≈ 0.3')
axes[0,0].set_title('k-distance график\n(подбор eps)', fontweight='bold')
axes[0,0].set_xlabel('Точки (отсортировано)')
axes[0,0].set_ylabel('5-е расстояние до соседа')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# DBSCAN с разными eps
params = [(0.15, 5), (0.3, 5), (0.6, 5), (0.3, 3), (0.3, 15)]
axes_flat = [axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]]

for ax, (eps, min_s) in zip(axes_flat, params):
    db = DBSCAN(eps=eps, min_samples=min_s)
    labels_db = db.fit_predict(X)
    n_cl = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()
    
    unique_labels = set(labels_db)
    colors_db = plt.cm.tab10(np.linspace(0, 0.9, max(n_cl, 1)))
    
    # Шум — серые точки
    noise_mask = labels_db == -1
    if noise_mask.sum() > 0:
        ax.scatter(X[noise_mask,0], X[noise_mask,1], c='gray',
                  alpha=0.3, s=25, marker='x', label=f'Шум ({n_noise})')
    
    ci = 0
    for lbl in sorted(unique_labels):
        if lbl == -1: continue
        mask = labels_db == lbl
        ax.scatter(X[mask,0], X[mask,1], color=colors_db[ci],
                  alpha=0.7, s=40, edgecolors='white', lw=0.3,
                  label=f'C{lbl+1} ({mask.sum()})')
        ci += 1
    
    sil_str = ''
    if n_cl > 1 and n_noise < len(labels_db):
        try:
            sil_str = f'  Sil={silhouette_score(X[labels_db != -1], labels_db[labels_db != -1]):.3f}'
        except: pass
    
    ax.set_title(f'eps={eps}, min_samples={min_s}\n→ {n_cl} кл, шум={n_noise}{sil_str}', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle(' DBSCAN: влияние параметров eps и min_samples', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

db_best = DBSCAN(eps=0.3, min_samples=5).fit(X)
lb = db_best.labels_
n_cl = len(set(lb)) - (1 if -1 in lb else 0)
print(f'\nDBSCAN (eps=0.3, min_samples=5):')
print(f'  Кластеров:  {n_cl}')
print(f'  Выбросов:   {(lb == -1).sum()}')
print()
print('   DBSCAN выявляет аномальные приёмы (выбросы) — нетипично дорогие')
print('   случаи или ошибки данных. Ценно для контроля качества клиники.')

---
## Метод 7: HDBSCAN (Hierarchical DBSCAN)

### Математика
HDBSCAN расширяет DBSCAN, строя **иерархию** кластеров переменной плотности:

1. Вычисляем **core distance** для каждой точки: расстояние до k-го соседа
2. Строим **mutual reachability distance**:  
   $d_{mreach}(a,b) = \max(core_k(a),\ core_k(b),\ d(a,b))$
3. Строим **минимальное остовное дерево** (MST) по этим расстояниям
4. Строим **иерархию кластеров** из MST
5. **Конденсируем дерево** по `min_cluster_size` — отсекаем нестабильные ветви
6. Выбираем **стабильные кластеры** по максимуму $\sum \lambda$ (lambda-persistence)

### Преимущества над DBSCAN:
- Не нужно подбирать `eps`
- Справляется с кластерами **разной плотности**
- Даёт вероятности принадлежности (`probabilities_`)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Различные min_cluster_size
mcs_list = [5, 15, 30, 50, 80]
for ax, mcs in zip(axes.flat[:5], mcs_list):
    hdb = HDBSCAN(min_cluster_size=mcs, min_samples=3)
    labels_hdb = hdb.fit_predict(X)
    n_cl = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    n_noise = (labels_hdb == -1).sum()
    
    unique = sorted(set(labels_hdb))
    colors_h = plt.cm.tab10(np.linspace(0, 0.9, max(n_cl, 1)))
    
    noise_mask = labels_hdb == -1
    if noise_mask.sum() > 0:
        ax.scatter(X[noise_mask,0], X[noise_mask,1], c='silver',
                  alpha=0.4, s=20, marker='x')
    ci = 0
    for lbl in unique:
        if lbl == -1: continue
        mask = labels_hdb == lbl
        # Цвет по вероятности принадлежности
        probs = hdb.probabilities_[mask]
        ax.scatter(X[mask,0], X[mask,1], color=colors_h[ci],
                  alpha=np.clip(probs, 0.2, 1.0), s=40,
                  edgecolors='white', lw=0.3, label=f'C{lbl+1} n={mask.sum()}')
        ci += 1
    
    ax.set_title(f'min_cluster_size={mcs}\n→ {n_cl} кл, шум={n_noise}', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Вероятности принадлежности
hdb_best = HDBSCAN(min_cluster_size=20, min_samples=5)
labels_hdb_b = hdb_best.fit_predict(X)
probs = hdb_best.probabilities_

n_cl = len(set(labels_hdb_b)) - (1 if -1 in labels_hdb_b else 0)
if n_cl >= 1:
    sc_plot = axes.flat[5].scatter(X[:,0], X[:,1], c=probs, cmap='RdYlGn',
                                   s=40, alpha=0.8, edgecolors='white', lw=0.3)
    plt.colorbar(sc_plot, ax=axes.flat[5], label='Вероятность принадлежности')
    axes.flat[5].set_title(f'HDBSCAN: вероятности\n(min_cluster_size=20, {n_cl} кл)', fontweight='bold')
    axes.flat[5].grid(True, alpha=0.3)

plt.suptitle(' HDBSCAN: иерархический DBSCAN', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nHDBSCAN (min_cluster_size=20):')
print(f'  Кластеров: {n_cl}')
print(f'  Выбросов:  {(labels_hdb_b == -1).sum()}')
print(f'  Уверенных точек (prob>0.8): {(probs > 0.8).sum()}')
print()
print('   HDBSCAN даёт вероятность принадлежности — полезно для стоматологии:')
print('   «пограничные» пациенты (вероятность 0.4–0.6) требуют ручной проверки.')


## Метод 8: OPTICS (Ordering Points To Identify Clustering Structure)

### Математика
OPTICS строит **упорядоченный список** точек по достижимости и создаёт **reachability plot**:

- **Core distance** $cd(o)$: минимальный радиус, при котором $o$ ядровая
- **Reachability distance** $rd(p, o)$:
$$rd(p, o) = \max(cd(o),\ d(o, p))$$

Точки упорядочены так, что **низкие долины** на reachability plot = **кластеры**, **высокие пики** = **граница кластеров**.

### Отличие от DBSCAN:
OPTICS не требует фиксированного `eps` — обрабатывает данные переменной плотности, строя единую иерархическую структуру.

In [ ]:
optics = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.1)
optics.fit(X)

labels_opt = optics.labels_
reach = optics.reachability_[optics.ordering_]
n_cl = len(set(labels_opt)) - (1 if -1 in labels_opt else 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Reachability Plot ────────────────────────────────────────────────────────
space = np.arange(len(reach))
labels_ord = labels_opt[optics.ordering_]
unique_lbl = sorted(set(labels_ord))
colors_op = plt.cm.tab10(np.linspace(0, 0.9, max(n_cl, 1)))

ci = 0
for lbl in unique_lbl:
    mask = labels_ord == lbl
    if lbl == -1:
        axes[0].bar(space[mask], reach[mask], width=1.0, color='gray', alpha=0.4)
    else:
        axes[0].bar(space[mask], reach[mask], width=1.0,
                   color=colors_op[ci], alpha=0.8, label=f'Кластер {lbl+1}')
        ci += 1

axes[0].set_ylabel('Reachability Distance')
axes[0].set_xlabel('Порядок обхода точек')
axes[0].set_title('Reachability Plot\n(долины = кластеры)', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.2)

# ── Результат кластеризации ──────────────────────────────────────────────────
noise_mask = labels_opt == -1
if noise_mask.sum() > 0:
    axes[1].scatter(X[noise_mask,0], X[noise_mask,1], c='lightgray',
                   alpha=0.4, s=20, marker='x', label=f'Шум ({noise_mask.sum()})')
ci = 0
for lbl in sorted(set(labels_opt)):
    if lbl == -1: continue
    mask = labels_opt == lbl
    axes[1].scatter(X[mask,0], X[mask,1], color=colors_op[ci],
                   alpha=0.7, s=40, edgecolors='white', lw=0.3,
                   label=f'Кластер {lbl+1} (n={mask.sum()})')
    ci += 1
axes[1].set_title(f'OPTICS результат\n→ {n_cl} кластеров', fontweight='bold')
axes[1].set_xlabel('Зубы (норм.)'); axes[1].set_ylabel('Стоимость (норм.)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# ── Сравнение методов извлечения кластеров из OPTICS ─────────────────────────
from sklearn.cluster import OPTICS
for xi_val, color, label_txt in [(0.03, PALETTE[0], 'xi=0.03'), 
                                  (0.07, PALETTE[1], 'xi=0.07'),
                                  (0.15, PALETTE[2], 'xi=0.15')]:
    op_xi = OPTICS(min_samples=10, xi=xi_val, min_cluster_size=0.08)
    op_xi.fit(X)
    n_xi = len(set(op_xi.labels_)) - (1 if -1 in op_xi.labels_ else 0)
    axes[2].scatter([], [], c=color, label=f'{label_txt} → {n_xi} кл')

# Отображаем лучший
ci = 0
for lbl in sorted(set(labels_opt)):
    if lbl == -1:
        axes[2].scatter(X[labels_opt==-1,0], X[labels_opt==-1,1],
                       c='silver', alpha=0.3, s=20, marker='x')
        continue
    mask = labels_opt == lbl
    axes[2].scatter(X[mask,0], X[mask,1], color=colors_op[ci],
                   alpha=0.7, s=40, edgecolors='white', lw=0.3)
    ci += 1
axes[2].set_title('Влияние параметра xi', fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.suptitle('🔭 OPTICS: упорядочение точек для структуры кластеров', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nOPTICS (min_samples=10, xi=0.05):')
print(f'  Найдено кластеров: {n_cl}')
print(f'  Точек-шума:        {(labels_opt == -1).sum()}')
print()
print('   Ключевая особенность OPTICS — Reachability Plot: визуально')
print('   видно, где начинаются и заканчиваются кластеры (долины на графике).')

---
## Метод 9: BIRCH (Balanced Iterative Reducing and Clustering using Hierarchies)

### Математика
BIRCH строит **CF-дерево** (Clustering Feature Tree) для эффективной работы с большими данными:

**Clustering Feature (CF)** для кластера из $N$ точек $\{x_1, ..., x_N\}$:
$$CF = (N,\ LS,\ SS)$$
- $N$ — число точек
- $LS = \sum x_i$ — линейная сумма
- $SS = \sum x_i^2$ — квадратичная сумма

**Диаметр кластера** (мера компактности):
$$D = \sqrt{\frac{2N \cdot SS - 2 \cdot LS^2}{N(N-1)}}$$

### Параметры:
- `threshold` — максимальный диаметр подкластера в листе CF-дерева
- `branching_factor` — максимальное число дочерних узлов
- `n_clusters` — финальное число кластеров (опционально)

### Преимущество: O(N) по памяти — работает на миллионах записей!

In [ ]:
# BIRCH на большом датасете — демонстрация масштабируемости
np.random.seed(42)
N_large = 5000
g1_l = np.random.multivariate_normal([1.2, 1500], [[0.15,200],[200,300000]], 1800)
g2_l = np.random.multivariate_normal([2.5, 5500], [[0.2,100],[100,600000]], 2000)
g3_l = np.random.multivariate_normal([3.8,11000], [[0.2,50],[50,800000]], 1200)
X_large_raw = np.vstack([g1_l, g2_l, g3_l])
X_large = StandardScaler().fit_transform(X_large_raw)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Разные threshold
thresholds = [0.1, 0.3, 0.7]
for ax, thr in zip(axes[0], thresholds):
    birch = Birch(threshold=thr, branching_factor=50, n_clusters=3)
    labels_birch = birch.fit_predict(X_large)
    n_cl = len(np.unique(labels_birch))
    sil = silhouette_score(X_large[::10], labels_birch[::10])  # на подвыборке
    for i in range(n_cl):
        mask = labels_birch == i
        ax.scatter(X_large[mask,0][::5], X_large[mask,1][::5],
                  c=PALETTE[i], alpha=0.4, s=10, edgecolors='none')
    ax.set_title(f'threshold={thr}\n→ {n_cl} кл | Sil={sil:.3f}', fontweight='bold')
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.grid(True, alpha=0.3)

# Масштабируемость
import time
sizes = [500, 1000, 2000, 5000, 10000, 20000]
times_birch, times_kmeans = [], []

for size in sizes:
    X_test = np.random.randn(size, 2)
    
    t0 = time.time()
    Birch(n_clusters=3).fit(X_test)
    times_birch.append(time.time() - t0)
    
    t0 = time.time()
    KMeans(n_clusters=3, n_init=3).fit(X_test)
    times_kmeans.append(time.time() - t0)

axes[1,0].plot(sizes, times_birch, 'o-', color=PALETTE[0], lw=2, ms=7, label='BIRCH')
axes[1,0].plot(sizes, times_kmeans, 's--', color=PALETTE[1], lw=2, ms=7, label='K-Means')
axes[1,0].set_xlabel('Размер выборки'); axes[1,0].set_ylabel('Время (сек)')
axes[1,0].set_title('BIRCH vs K-Means: масштабируемость', fontweight='bold')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# BIRCH без финального кластеризатора (только CF-дерево)
birch_nocl = Birch(threshold=0.3, n_clusters=None)
labels_sub = birch_nocl.fit_predict(X_large)
n_sub = len(np.unique(labels_sub))
axes[1,1].scatter(X_large[::4,0], X_large[::4,1],
                 c=[plt.cm.tab20(l % 20 / 20.0) for l in labels_sub[::4]],
                 alpha=0.4, s=10)
axes[1,1].set_title(f'BIRCH без финального k\n→ {n_sub} подкластеров (CF-листьев)', fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

# Итоговый лучший вариант
birch_best = Birch(threshold=0.3, branching_factor=50, n_clusters=3)
labels_birch_b = birch_best.fit_predict(X_large)
for i in range(3):
    mask = labels_birch_b == i
    axes[1,2].scatter(X_large[mask,0][::4], X_large[mask,1][::4],
                     c=PALETTE[i], alpha=0.5, s=10,
                     label=f'K{i+1} (n={mask.sum()})')
axes[1,2].set_title(f'BIRCH лучший: N={N_large} точек', fontweight='bold')
axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3)

plt.suptitle(' BIRCH: масштабируемая кластеризация', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nBIRCH на {N_large} записях:')
print(f'  Silhouette: {silhouette_score(X_large[::10], labels_birch_b[::10]):.4f}')
print()
print('   BIRCH идеален для роста клиники: миллионы записей пациентов')
print('   обрабатываются за O(N) памяти. Сжимает данные в CF-дерево.')

---
## Метод 10: Оценка производительности кластеризации

### Внутренние метрики (не нужны истинные метки)

**Calinski-Harabasz Index (CHI)**:
$$CH = \frac{\text{tr}(B_k)}{\text{tr}(W_k)} \cdot \frac{n - k}{k - 1}$$
- $B_k$ — дисперсия **между** кластерами, $W_k$ — **внутри**
- Чем выше — тем лучше (нет верхней границы)

**Davies-Bouldin Index (DBI)**:
$$DB = \frac{1}{k} \sum_{i=1}^k \max_{j \neq i} \frac{s_i + s_j}{d_{ij}}$$
- Чем **ниже** — тем лучше (0 — идеал)

### Внешние метрики (нужны истинные метки)
- **ARI** (Adjusted Rand Index): $[-1, 1]$, 1 = идеально, 0 = случайно
- **AMI** (Adjusted Mutual Information): похоже на ARI

In [ ]:
# ── Вычислим метрики для всех методов ────────────────────────────────────────
algorithms = {
    'K-Means':           KMeans(n_clusters=3, n_init=10, random_state=42),
    'Agglomerative':     AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'Spectral':          SpectralClustering(n_clusters=3, random_state=42, n_init=5),
    'BIRCH':             Birch(threshold=0.3, n_clusters=3),
    'Mean Shift':        MeanShift(bandwidth=estimate_bandwidth(X, quantile=0.3, n_samples=150, random_state=42)),
    'DBSCAN':            DBSCAN(eps=0.3, min_samples=5),
    'HDBSCAN':           HDBSCAN(min_cluster_size=20, min_samples=5),
    'OPTICS':            OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.1),
}

results = []
all_labels = {}

for name, algo in algorithms.items():
    labels = algo.fit_predict(X)
    all_labels[name] = labels
    
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    
    # Вычисляем метрики только если кластеров > 1
    valid = labels[labels != -1]
    X_valid = X[labels != -1]
    
    if n_cl > 1 and len(X_valid) > 10:
        sil = round(silhouette_score(X_valid, valid), 4)
        chi = round(calinski_harabasz_score(X_valid, valid), 2)
        dbi = round(davies_bouldin_score(X_valid, valid), 4)
        ari = round(adjusted_rand_score(true_labels[labels != -1], valid), 4)
        ami = round(adjusted_mutual_info_score(true_labels[labels != -1], valid), 4)
    else:
        sil = chi = dbi = ari = ami = None
    
    results.append({'Алгоритм': name, 'Кластеров': n_cl, 'Шума': n_noise,
                    'Silhouette↑': sil, 'Calinski-Harabasz↑': chi,
                    'Davies-Bouldin↓': dbi, 'ARI↑': ari, 'AMI↑': ami})

df_res = pd.DataFrame(results)
print('=' * 80)
print('СВОДНАЯ ТАБЛИЦА МЕТРИК КЛАСТЕРИЗАЦИИ')
print('=' * 80)
print(df_res.to_string(index=False))
print()
print('↑ = выше лучше | ↓ = ниже лучше')

# ── Визуализация сравнения ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_valid = df_res.dropna(subset=['Silhouette↑'])
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(df_valid))]

for ax, col, title, higher in zip(
    axes.flat,
    ['Silhouette↑', 'Calinski-Harabasz↑', 'Davies-Bouldin↓', 'ARI↑'],
    ['Silhouette Score (↑ лучше)', 'Calinski-Harabasz Index (↑ лучше)',
     'Davies-Bouldin Index (↓ лучше)', 'Adjusted Rand Index (↑ лучше)'],
    [True, True, False, True]
):
    vals = df_valid[col].astype(float)
    best_idx = vals.idxmax() if higher else vals.idxmin()
    bar_colors = ['gold' if i == best_idx else colors_bar[list(df_valid.index).index(i)]
                  for i in df_valid.index]
    bars = ax.bar(df_valid['Алгоритм'], vals, color=bar_colors,
                  edgecolor='white', lw=0.5, alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xticklabels(df_valid['Алгоритм'], rotation=30, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    # Значения над барами
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
               f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle(' Сравнение алгоритмов кластеризации\n( = лучший результат)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Итоговая визуализация всех методов на одном холсте ───────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes_flat = axes.flat

for ax, (name, labels) in zip(axes_flat, all_labels.items()):
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    unique = sorted(set(labels))
    n_col = max(n_cl, 1)
    colors_all = plt.cm.tab10(np.linspace(0, 0.9, n_col))
    
    noise_mask = labels == -1
    if noise_mask.sum() > 0:
        ax.scatter(X[noise_mask,0], X[noise_mask,1], c='silver',
                  alpha=0.3, s=15, marker='x')
    ci = 0
    for lbl in unique:
        if lbl == -1: continue
        mask = labels == lbl
        ax.scatter(X[mask,0], X[mask,1], color=colors_all[ci],
                  alpha=0.65, s=25, edgecolors='white', lw=0.2)
        ci += 1
    
    sil_str = ''
    row = df_res[df_res['Алгоритм'] == name]
    if not row.empty and row['Silhouette↑'].values[0] is not None:
        sil_str = f'\nSil={row["Silhouette↑"].values[0]:.3f}'
    ax.set_title(f'{name}\n→ {n_cl} кл{sil_str}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Зубы (норм.)', fontsize=8)
    ax.set_ylabel('Стоимость (норм.)', fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=7)

# Последняя ячейка — сводная таблица
ax_last = axes_flat[8]
ax_last.axis('off')
table_data = df_res[['Алгоритм', 'Кластеров', 'Силуэт']].copy()
cell_text = [[row['Алгоритм'], str(row['Кластеров']),
              str(row['Silhouette↑']) if row['Silhouette↑'] else 'N/A']
             for _, row in df_res.iterrows()]
tbl = ax_last.table(cellText=cell_text,
                    colLabels=['Алгоритм', 'Кластеров', 'Silhouette'],
                    cellLoc='center', loc='center',
                    bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2563EB')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F8FAFC')

plt.suptitle(' Все методы кластеризации: стоматологическая клиника\n'
             'scikit-learn | Python 3.11 | Задание 4',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('clustering_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Итоговый рисунок сохранён: clustering_all_methods.png')

In [ ]:
# ── Силуэтный анализ для K-Means ─────────────────────────────────────────────
from sklearn.metrics import silhouette_samples

km_sil = KMeans(n_clusters=3, n_init=10, random_state=42)
labels_sil = km_sil.fit_predict(X)
sil_vals = silhouette_samples(X, labels_sil)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Силуэтный график
y_lower = 10
for i in range(3):
    vals_i = np.sort(sil_vals[labels_sil == i])
    size_i = len(vals_i)
    y_upper = y_lower + size_i
    axes[0].fill_betweenx(np.arange(y_lower, y_upper), 0, vals_i,
                          facecolor=PALETTE[i], alpha=0.8, label=f'Кластер {i+1}')
    axes[0].text(-0.05, y_lower + size_i/2, f'C{i+1}', fontsize=9, va='center', fontweight='bold')
    y_lower = y_upper + 10

axes[0].axvline(x=silhouette_score(X, labels_sil), color='red', linestyle='--', lw=2,
               label=f'Avg Silhouette = {silhouette_score(X, labels_sil):.3f}')
axes[0].set_title('Силуэтный анализ K-Means (k=3)', fontweight='bold')
axes[0].set_xlabel('Silhouette coefficient')
axes[0].set_ylabel('Точки кластеров')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

# Итоговый рейтинг алгоритмов
df_rank = df_valid.sort_values('Silhouette↑', ascending=False).reset_index(drop=True)
df_rank['Ранг'] = range(1, len(df_rank)+1)
colors_rank = ['gold','silver','#CD7F32'] + [PALETTE[3]] * len(df_rank)

bars = axes[1].barh(df_rank['Алгоритм'][::-1], df_rank['Silhouette↑'][::-1],
                   color=colors_rank[:len(df_rank)][::-1],
                   edgecolor='white', lw=0.5, alpha=0.9)
axes[1].axvline(x=0.5, color='red', linestyle=':', lw=1.5, label='Хороший порог (0.5)')
axes[1].set_title('Рейтинг по Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Silhouette Score')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, df_rank['Silhouette↑'][::-1]):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
               f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

plt.suptitle(' Силуэтный анализ и итоговый рейтинг алгоритмов', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n' + '='*65)
print(' ИТОГОВЫЕ ВЫВОДЫ')
print('='*65)
conclusions = [
    ('K-Means',        ' Лучший для компактных сферических кластеров. Быстрый.'),
    ('Agglomerative',  ' Даёт дендрограмму — наглядно для врачей клиники.'),
    ('Spectral',       ' Для нелинейных данных. Медленнее, но точнее.'),
    ('BIRCH',          ' Масштабируется на миллионы записей. Для Big Data.'),
    ('HDBSCAN',        ' Лучший density-метод. Выдаёт вероятности.'),
    ('DBSCAN',         ' Хорош для выявления аномальных приёмов (выбросов).'),
    ('OPTICS',         ' Reachability plot — полезен для визуального анализа.'),
    ('Mean Shift',     ' Автоматически находит k. Чувствителен к bandwidth.'),
    ('Aff. Prop.',     ' Находит exemplars — «типичные» случаи лечения.'),
]
for algo, desc in conclusions:
    print(f'  {algo:15s}: {desc}')
print('='*65)
print('\n Рекомендация для стоматологической клиники:')
print('   K-Means(k=3) → выявление типов лечения (профилактика / плановое / сложное)')
print('   HDBSCAN      → выявление нетипичных и аномальных приёмов')
print('   BIRCH        → аналитика при росте клиники (>100k записей)')

---
## Итоговая сводка

| Алгоритм | Задаёт k? | Выбросы | Форма кластеров | Лучший случай |
|---|---|---|---|---|
| **K-Means** | Да | Нет | Сферическая | Компактные равные кластеры |
| **Affinity Propagation** | Нет | Нет | Любая | Нужны exemplars |
| **Mean Shift** | Нет | Нет | Выпуклая | Известна bandwidth |
| **Spectral** | Да | Нет | **Произвольная** | Нелинейные данные |
| **Agglomerative** | Да | Нет | Любая | Нужна дендрограмма |
| **DBSCAN** | Нет | **Да** | **Произвольная** | Выбросы важны |
| **HDBSCAN** | Нет | **Да** | **Произвольная** | Разная плотность кластеров |
| **OPTICS** | Нет | **Да** | **Произвольная** | Визуализация структуры |
| **BIRCH** | Опционально | Нет | Выпуклая | **Большие данные (Big Data)** |